# vlm — a vision-language model as a detection

Runs the example config `example/vlm/cap_check.yaml` against a hosted
vision-language model: one image, several views of one part, and a
per-call extra reference. Contract: workspace `docs/vision-guide.md` §9.

**Before running**
- The vision server on this host runs the `pro_vlm` code (restart it from this checkout).
- This vision unit can reach the internet.
- Your key is in `example/vlm/vlm.key`, beside the config — just the key, nothing
  else; the config's `key_path: vlm.key` names it, and `*.key` is git-ignored.
  Create it once, from `example/vlm/`:

      echo 'PASTE-YOUR-KEY' > vlm.key && chmod 600 vlm.key

  (Or, for a quick test only, write it into the config's `key:` — and never commit that.)

## 1. Parameters

In [1]:
HOST, PORT = "127.0.0.1", 4001
CONFIG = "../vlm/cap_check.yaml"     # the example config file (references beside it in refs/)
IMAGE  = "../img/tube.jpg"           # the image to judge
OTHER  = "../img/vial.jpg"           # a second view, for the multi-view call

## 2. Connect and add the detection

The config file is read here, its reference images shipped with the call. The key is the config's `key`, or else read from its `key_path` on this computer and sent as its own argument; the server keeps it in memory for this session only. The reply's `config` is the merged result.

In [ ]:
from dorna_vision_client import VisionClient

vc = VisionClient()
vc.connect(host=HOST, port=PORT)
reply = vc.detection_add("cap", config=CONFIG)
print(reply["cmd"], "—", reply["config"]["detection"]["labels"])
det = vc.detection("cap")
det.detection()

## 3. Judge one image

`views=[...]` sends images from this computer — no camera. The answer comes back in the usual detection shape, plus `reason` and `vlm` (model, latency, tokens).

In [ ]:
def show(res):
    for e in res:
        v = e["vlm"]
        print(f"{e['cls']!r}  conf {e['conf']:.2f}  — {e['reason']}")
        print(f"   {v['model']} · {v['latency_ms']} ms · tokens in {v['tokens_in']} "
              f"(cached {v['tokens_cached']}) · out {v['tokens_out']}")
        print(e)
show(det.run(views=[IMAGE]))

## 4. Several views of ONE part

One request, one verdict for the part, judged across every view.

In [ ]:
show(det.run(views=[IMAGE, OTHER]))

## 5. A per-call reference and prompt

Added after the config's own references and prompt — that part stays the same, so the provider can cache it.

In [ ]:
show(det.run(views=[IMAGE],
             extra_refs=[{"image": OTHER, "label": "uncapped", "note": "an open vial: no cap at all"}],
             prompt="Judge only the tube nearest the centre."))

## 6. The log

`display.client_save_log` (set in `cap_check.yaml`): every call's views and its answer JSON are written on this computer, under `example/vlm/output/` — beside the config: a relative path in a config is relative to the config, and `true` is the folder `output/` (a path like `"~/captures/cap_check/"` writes there instead).

In [ ]:
import os
d = os.path.expanduser("~/captures/cap_check")
print(sorted(os.listdir(d))[-6:] if os.path.isdir(d) else "no log yet")

## 7. (Optional) A live camera, several poses

With a camera on the unit: add the detection on it, keep a view per pose with `add_view()`, and `run()` sends them all at once.

In [ ]:
# SERIAL = "..."
# vc.detection_add("cap_live", camera_serial_number=SERIAL,
#                  config=CONFIG)
# live = vc.detection("cap_live")
# live.add_view()        # pose 1
# live.add_view()        # pose 2
# show(live.run())       # pose 3 + the two kept views, one request

## 8. Clean up

In [ ]:
vc.detection_remove("cap")
vc.close()